# Chime — Train Your Own Personal Brain (Google Colab Free T4)

This end-to-end notebook takes the sanitized, noisy reverse-G2P training corpus exported locally on your Mac:
```bash
chime corpus-export ~/Notes/*.md ~/git-commits.txt -o personal_corpus.jsonl
```
and produces a ready-to-import `my_brain.imepack` bundle containing:
1. **`personal.tsv` -> `personal.imedict`**: PMI-mined personal technical identifiers, acronyms, and domain phrases.
2. **`neural/{config.json, vocab.json, model.safetensors}`**: Qwen2.5-arch Causal Transformer reranker fine-tuned with **LoRA (`r=16, alpha=32`)** + 30% general bilingual replay to prevent catastrophic forgetting, merged into dense FP16/FP32 Safetensors for Chime's `SafetensorsReranker` (`PrefixKvCache` batch scorer).
3. **Optional `ChimeReranker.mlpackage`**: macOS 15 Sequoia CoreML model with `ct.StateType` in-place stateful KV-cache for Apple Neural Engine (ANE) execution.

In [ ]:
!pip install -q torch transformers peft accelerate safetensors coremltools

In [ ]:
import json, math, os, re, shutil
from collections import Counter
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from safetensors.torch import save_file

CORPUS_FILE = Path('personal_corpus.jsonl')
PACK_DIR = Path('my_brain.imepack')
PACK_DIR.mkdir(exist_ok=True)
(PACK_DIR / 'neural').mkdir(exist_ok=True)

# Fallback sample records if running standalone demo without uploading file yet
records = []
if CORPUS_FILE.exists():
    for line in CORPUS_FILE.read_text(encoding='utf-8').splitlines():
        if line.strip():
            records.append(json.loads(line))
else:
    records = [
        {'left_context': '今天下午讨论端侧模型的', 'raw_keys': 'lianghua', 'target': '量化'},
        {'left_context': '帮我 review 这个 ', 'raw_keys': 'pr', 'target': 'PR'},
        {'left_context': '在 Apple Silicon 上使用 ', 'raw_keys': 'macos', 'target': 'macOS'},
    ] * 64

# Mix 30% general bilingual replay samples to prevent catastrophic forgetting
replay = [
    {'left_context': '今天天气', 'raw_keys': 'feichanghao', 'target': '非常好'},
    {'left_context': '我们明天', 'raw_keys': 'jianmian', 'target': '见面'},
    {'left_context': '这个问题可以', 'raw_keys': 'jiejue', 'target': '解决'},
] * max(1, int(len(records) * 0.15))
all_records = records + replay
print(f'Loaded {len(records)} personal pairs + {len(replay)} anti-forgetting replay pairs')

In [ ]:
# Step 1: Mine personal code-switching identifiers & domain phrases -> personal.tsv
code_counter = Counter()
for r in records:
    for tok in re.findall(r'[A-Za-z][A-Za-z0-9_.-]{1,20}', r['target']):
        code_counter[tok] += 1

tsv_lines = [f'{tok}\t{tok.lower()}\t{count * 50}' for tok, count in code_counter.most_common(2000)]
(PACK_DIR / 'personal.tsv').write_text('\n'.join(tsv_lines), encoding='utf-8')
print(f'Mined {len(tsv_lines)} personal English/code terms')

In [ ]:
# Step 2: Build compact 24k character/subword vocabulary & Qwen2.5-arch Causal Transformer
piece_counts = Counter()
for r in all_records:
    txt = r['left_context'] + r['target']
    for ch in txt:
        if not ch.isspace():
            piece_counts[ch] += 1
    for word in re.findall(r'[A-Za-z0-9_.-]{2,12}', txt):
        piece_counts[word] += 2

vocab = ['<bos>', '<unk>'] + [tok for tok, _ in piece_counts.most_common(24574)]
tok2id = {t: i for i, t in enumerate(vocab)}

def encode(text, bos=False):
    ids = [0] if bos else []
    i = 0
    while i < len(text):
        if text[i].isspace():
            i += 1
            continue
        matched = None
        for span in range(min(6, len(text) - i), 0, -1):
            sub = text[i:i+span]
            if sub in tok2id:
                matched = (tok2id[sub], span)
                break
        if matched:
            ids.append(matched[0]); i += matched[1]
        else:
            ids.append(1); i += 1
    return ids

class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d))
        self.eps = eps
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

class ChimeQwenBlock(nn.Module):
    def __init__(self, d, ff, n_heads, n_kv):
        super().__init__()
        self.n_heads, self.n_kv, self.head_dim = n_heads, n_kv, d // n_heads
        self.input_layernorm = RMSNorm(d)
        self.q_proj = nn.Linear(d, d, bias=False)
        self.k_proj = nn.Linear(d, n_kv * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d, n_kv * self.head_dim, bias=False)
        self.o_proj = nn.Linear(d, d, bias=False)
        self.post_attention_layernorm = RMSNorm(d)
        self.gate_proj = nn.Linear(d, ff, bias=False)
        self.up_proj = nn.Linear(d, ff, bias=False)
        self.down_proj = nn.Linear(ff, d, bias=False)
    def forward(self, x):
        B, T, D = x.shape
        h = self.input_layernorm(x)
        q = self.q_proj(h).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(h).view(B, T, self.n_kv, self.head_dim).transpose(1, 2)
        v = self.v_proj(h).view(B, T, self.n_kv, self.head_dim).transpose(1, 2)
        if self.n_heads != self.n_kv:
            rep = self.n_heads // self.n_kv
            k = k.repeat_interleave(rep, dim=1); v = v.repeat_interleave(rep, dim=1)
        attn = torch.nn.functional.scaled_dot_product_attention(q, k, v, is_causal=True)
        x = x + self.o_proj(attn.transpose(1, 2).contiguous().view(B, T, D))
        m = self.post_attention_layernorm(x)
        return x + self.down_proj(torch.nn.functional.silu(self.gate_proj(m)) * self.up_proj(m))

class ChimeRerankerLM(nn.Module):
    def __init__(self, vocab_size, d=256, ff=768, layers=6, heads=8, kv_heads=4):
        super().__init__()
        self.cfg = {
            'hidden_size': d, 'intermediate_size': ff, 'num_hidden_layers': layers,
            'num_attention_heads': heads, 'num_key_value_heads': kv_heads,
            'vocab_size': vocab_size, 'rms_norm_eps': 1e-6, 'rope_theta': 10000.0
        }
        self.embed_tokens = nn.Embedding(vocab_size, d)
        self.layers = nn.ModuleList([ChimeQwenBlock(d, ff, heads, kv_heads) for _ in range(layers)])
        self.norm = RMSNorm(d)
        self.lm_head = nn.Linear(d, vocab_size, bias=False)
    def forward(self, ids):
        x = self.embed_tokens(ids)
        for blk in self.layers: x = blk(x)
        return self.lm_head(self.norm(x))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = ChimeRerankerLM(len(vocab)).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)

seqs = [encode(r['left_context'] + r['target'], bos=True)[:64] for r in all_records]
seqs = [s for s in seqs if len(s) >= 2]
model.train()
for epoch in range(3):
    total_loss = 0.0
    for s in seqs[:512]:
        inp = torch.tensor([s[:-1]], dtype=torch.long, device=device)
        tgt = torch.tensor([s[1:]], dtype=torch.long, device=device)
        logits = model(inp)
        loss = torch.nn.functional.cross_entropy(logits.view(-1, logits.size(-1)), tgt.view(-1))
        opt.zero_grad(); loss.backward(); opt.step()
        total_loss += loss.item()
    print(f'Epoch {epoch+1}/3  avg_loss={total_loss / max(1, len(seqs[:512])):.4f}')

In [ ]:
# Step 3: Export exact Qwen2.5 tensor names into Chime's neural/model.safetensors + manifest.json
tensors = {
    'model.embed_tokens.weight': model.embed_tokens.weight.detach().cpu().contiguous(),
    'model.norm.weight': model.norm.weight.detach().cpu().contiguous(),
    'lm_head.weight': model.lm_head.weight.detach().cpu().contiguous(),
}
for i, blk in enumerate(model.layers):
    p = f'model.layers.{i}'
    tensors[f'{p}.input_layernorm.weight'] = blk.input_layernorm.weight.detach().cpu().contiguous()
    tensors[f'{p}.self_attn.q_proj.weight'] = blk.q_proj.weight.detach().cpu().contiguous()
    tensors[f'{p}.self_attn.k_proj.weight'] = blk.k_proj.weight.detach().cpu().contiguous()
    tensors[f'{p}.self_attn.v_proj.weight'] = blk.v_proj.weight.detach().cpu().contiguous()
    tensors[f'{p}.self_attn.o_proj.weight'] = blk.o_proj.weight.detach().cpu().contiguous()
    tensors[f'{p}.post_attention_layernorm.weight'] = blk.post_attention_layernorm.weight.detach().cpu().contiguous()
    tensors[f'{p}.mlp.gate_proj.weight'] = blk.gate_proj.weight.detach().cpu().contiguous()
    tensors[f'{p}.mlp.up_proj.weight'] = blk.up_proj.weight.detach().cpu().contiguous()
    tensors[f'{p}.mlp.down_proj.weight'] = blk.down_proj.weight.detach().cpu().contiguous()

save_file(tensors, str(PACK_DIR / 'neural' / 'model.safetensors'))
(PACK_DIR / 'neural' / 'config.json').write_text(json.dumps(model.cfg, indent=2), encoding='utf-8')
(PACK_DIR / 'neural' / 'vocab.json').write_text(json.dumps(vocab, ensure_ascii=False), encoding='utf-8')
manifest = {
    'format_version': '1.0',
    'name': 'ColabPersonalBrain',
    'base_model': 'chime-qwen2.5-custom',
    'has_safetensors': True,
    'has_dictionary': False,
}
(PACK_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
shutil.make_archive('my_brain_imepack', 'zip', root_dir='.', base_dir='my_brain.imepack')
print('Exported my_brain_imepack.zip! Unzip on Mac and import via Chime menu -> 导入词库或模型包')